# Model Optimization: Knowledge Distillation

In this notebook, we'll apply knowledge distillation to our models using distributed processing. Knowledge distillation is a technique where a smaller "student" model is trained to mimic the behavior of a larger "teacher" model.

## What is Knowledge Distillation?

Knowledge distillation is a model compression technique where a small model (student) is trained to mimic a larger, more complex model (teacher). The key insight is that the student model learns from the teacher's soft probability outputs rather than just the hard labels, allowing it to capture the nuanced "dark knowledge" embedded in the teacher's predictions.

### Benefits of Knowledge Distillation:
- **Smaller Model Size**: Student models have fewer parameters
- **Faster Inference**: Smaller models require less computation
- **Lower Memory Requirements**: Reduced memory footprint
- **Preserved Accuracy**: Often maintains most of the teacher model's accuracy

### How Knowledge Distillation Works:
1. **Teacher Model**: A large, pre-trained model with high accuracy
2. **Student Model**: A smaller model with fewer parameters
3. **Soft Targets**: The teacher's probability distributions (softened with temperature)
4. **Training**: The student is trained to match both the correct labels and the teacher's soft targets

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform knowledge distillation on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
import json
import time
import pandas as pd
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
import time
from IPython.display import clear_output

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    SAGEMAKER_ROLE_ARN = "YOUR_ROLE_ARN_HERE"  # Update this value
    OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Default optimization instance type
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN
    %store OPTIMIZATION_INSTANCE_TYPE

## 3. Load Model Information

In [ ]:
# Try to load baseline metrics if they exist
try:
    with open('baseline_metrics.json', 'r') as f:
        baseline_metrics = json.load(f)
    print(f"Loaded baseline metrics for {len(baseline_metrics)} models")
except FileNotFoundError:
    print("baseline_metrics.json not found. Will proceed without baseline metrics.")
    baseline_metrics = {}

# Try to load model information from file
try:
    with open('model_info.json', 'r') as f:
        model_info = json.load(f)
    print(f"Loaded information for {len(model_info)} models")
except FileNotFoundError:
    print("model_info.json not found. Using default model information.")
    model_info = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }

## 4. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment-analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question-answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked-lm": "The [MASK] is a large language model trained by OpenAI."
}

## 5. Create Distillation Script

In this section, we'll create a Python script that performs the actual knowledge distillation. This script will be executed on the SageMaker Processing instances.

### What the Script Does:
1. **Loads the teacher model** from Hugging Face or S3
2. **Creates a smaller student model** with the same architecture but fewer layers
3. **Prepares a dataset** for distillation (using a subset of the original training data)
4. **Trains the student model** using both hard labels and soft targets from the teacher
5. **Measures performance metrics** like model size and inference time
6. **Saves the distilled model** and metrics to the output directory

### Distillation Parameters:
- **Temperature**: Controls the softness of the teacher's probability distributions (higher = softer)
- **Alpha**: Weight between hard label loss and soft target loss
- **Epochs**: Number of training epochs
- **Batch Size**: Number of samples per batch

The script will be stored in a dedicated directory to keep our project organized.

In [ ]:
# Check if the distillation_scripts directory exists, if not create it
import os
if not os.path.exists('distillation_scripts'):
    os.makedirs('distillation_scripts')
    print("Created distillation_scripts directory")
else:
    print("distillation_scripts directory already exists")

## 6. Launch Distributed Distillation Jobs

Now we'll set up and launch the SageMaker Processing jobs to perform knowledge distillation. Each model will be processed in a separate job, allowing for parallel processing.

### Distillation Process:
1. **Create a PyTorch processor** with the appropriate instance type and configuration
2. **For each model**:
   - Save and upload model information to S3
   - Define inputs (distillation script and model info) and outputs
   - Launch a processing job with the appropriate arguments
   - Store the job information for monitoring

We're using a temperature of 2.0, which is a common choice for knowledge distillation. Higher temperatures make the teacher's probability distributions softer, helping the student learn the relationships between classes.

In [ ]:
# Define the instance type to use for distillation
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="1.13.1",
    py_version="py39",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="model-distillation",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch distillation jobs for each model
distillation_jobs = {}
job_output_paths = {}
s3_client = boto3.client('s3')

for model_key in model_info.keys():
    # Skip models that are not suitable for distillation
    if model_info[model_key]["task"] not in ["sequence-classification", "token-classification"]:
        print(f"Skipping {model_key} as it's not suitable for distillation")
        continue
        
    print(f"\nLaunching distillation job for {model_key}...")
    
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define the output path
    output_path = f's3://{S3_BUCKET}/optimization/outputs/{model_key}_distilled'
    job_output_paths[model_key] = output_path
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data/model_info.json'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            output_name='distilled_model',
            source='/opt/ml/processing/output',
            destination=output_path
        )
    ]
    
    # Run the processing job
    try:
        distillation_jobs[model_key] = processor.run(
            source_dir='distillation_scripts',
            code='distillation_script.py',
            inputs=inputs,
            outputs=outputs,
            arguments=[
                '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
                '--output-dir', '/opt/ml/processing/output',
                '--epochs', '3',
                '--batch-size', '8',
                '--temperature', '2.0'
            ]
        )
        # No print statement here that could cause errors
    except Exception as e:
        print(f"Error launching job for {model_key}: {e}")
        distillation_jobs[model_key] = None

## 7. Collect Results

Now that the distillation jobs are complete, we'll collect and combine the results from each job. Each job produces a metrics file containing information about the distilled model, such as size, inference time, and the comparison with the teacher model.

### Collection Process:
1. **Download metrics files** from S3 for each model
2. **Combine metrics** into a single dictionary
3. **Save combined metrics** to a local file for use in later notebooks

This gives us a comprehensive view of the distillation results across all models, which we'll analyze in the next section.

In [ ]:
# Download and combine results
distilled_metrics = {}

for model_key in model_info.keys():
    # Check if we have an output path for this model
    if model_key not in job_output_paths:
        print(f"No output path found for {model_key}, skipping metrics collection")
        continue
        
    # Download metrics file
    try:
        # Use the saved output path
        s3_client.download_file(
            S3_BUCKET,
            f'optimization/outputs/{model_key}_distilled/distilled_metrics.json',
            f'temp_{model_key}_distilled_metrics.json'
        )
        
        # Load metrics
        with open(f'temp_{model_key}_distilled_metrics.json', 'r') as f:
            metrics = json.load(f)
        
        # Add to combined metrics
        distilled_metrics.update(metrics)
        
        print(f"Downloaded metrics for {model_key}")
    except Exception as e:
        print(f"Error downloading metrics for {model_key}: {e}")
        # Continue with other models even if one fails
        continue

# Save combined metrics
with open('distilled_metrics.json', 'w') as f:
    json.dump(distilled_metrics, f, indent=2)

print(f"\nSaved distilled metrics for {len(distilled_metrics)} models to distilled_metrics.json")

## 8. Compare Results

Now we'll compare the performance of the distilled student models against the original teacher models. This comparison helps us understand the impact of knowledge distillation on model size and inference speed.

### Key Metrics to Compare:
- **Model Size**: How much smaller are the student models?
- **Inference Time**: How much faster are the student models?
- **Memory Usage**: How much less memory do the student models use?
- **Size Reduction Percentage**: The percentage reduction in model size
- **Inference Speedup Percentage**: The percentage improvement in inference speed

We expect to see significant size reductions and inference speedups with minimal impact on accuracy. Knowledge distillation typically achieves better results than pruning or quantization alone, especially for complex tasks.

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

for model_key in distilled_metrics.keys():
    # Get teacher and student metrics
    teacher = distilled_metrics[model_key]['teacher']
    student = distilled_metrics[model_key]['student']
    
    # Calculate improvements
    size_reduction = (teacher['model_size'] - student['model_size']) / teacher['model_size'] * 100
    time_reduction = (teacher['inference_time'] - student['inference_time']) / teacher['inference_time'] * 100
    memory_reduction = (teacher['memory_usage'] - student['memory_usage']) / teacher['memory_usage'] * 100
    
    comparison_data.append({
        'Model': student['model_name'],
        'Teacher Model': teacher['model_name'],
        'Student Model': student['student_model_name'],
        'Teacher Size (MB)': teacher['model_size'],
        'Student Size (MB)': student['model_size'],
        'Size Reduction (%)': size_reduction,
        'Teacher Inference (ms)': teacher['inference_time'],
        'Student Inference (ms)': student['inference_time'],
        'Inference Speedup (%)': time_reduction,
        'Teacher Memory (MB)': teacher['memory_usage'],
        'Student Memory (MB)': student['memory_usage'],
        'Memory Reduction (%)': memory_reduction
    })

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df

## 9. Visualize Results

Let's create some visualizations to better understand the impact of knowledge distillation on our models. We'll create bar charts comparing the teacher and student models across different metrics.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style
sns.set(style="whitegrid")

# Create a figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot model size comparison
for i, model in enumerate(comparison_data):
    axes[0].bar([i-0.2, i+0.2], [model['Teacher Size (MB)'], model['Student Size (MB)']], width=0.4, 
             color=['#1f77b4', '#ff7f0e'], label=['Teacher', 'Student'] if i == 0 else None)
    
axes[0].set_title('Model Size Comparison')
axes[0].set_ylabel('Size (MB)')
axes[0].set_xticks(range(len(comparison_data)))
axes[0].set_xticklabels([model['Model'] for model in comparison_data], rotation=45, ha='right')
axes[0].legend()

# Plot inference time comparison
for i, model in enumerate(comparison_data):
    axes[1].bar([i-0.2, i+0.2], [model['Teacher Inference (ms)'], model['Student Inference (ms)']], width=0.4, 
             color=['#1f77b4', '#ff7f0e'], label=['Teacher', 'Student'] if i == 0 else None)
    
axes[1].set_title('Inference Time Comparison')
axes[1].set_ylabel('Time (ms)')
axes[1].set_xticks(range(len(comparison_data)))
axes[1].set_xticklabels([model['Model'] for model in comparison_data], rotation=45, ha='right')
axes[1].legend()

# Plot memory usage comparison
for i, model in enumerate(comparison_data):
    axes[2].bar([i-0.2, i+0.2], [model['Teacher Memory (MB)'], model['Student Memory (MB)']], width=0.4, 
             color=['#1f77b4', '#ff7f0e'], label=['Teacher', 'Student'] if i == 0 else None)
    
axes[2].set_title('Memory Usage Comparison')
axes[2].set_ylabel('Memory (MB)')
axes[2].set_xticks(range(len(comparison_data)))
axes[2].set_xticklabels([model['Model'] for model in comparison_data], rotation=45, ha='right')
axes[2].legend()

plt.tight_layout()
plt.show()

## 10. Deploy Models to SageMaker for Inference

Now that we've created distilled student models, let's deploy them to SageMaker endpoints for real-world inference testing. We'll deploy both the original teacher models and the distilled student models to compare their performance.

### Deployment Process:
1. **Create model artifacts** in S3 for both teacher and student models
2. **Create SageMaker models** using these artifacts
3. **Deploy models to endpoints** for inference
4. **Test inference** with sample inputs
5. **Compare performance** between teacher and student models

This will give us a real-world comparison of the inference performance between the original and distilled models.

In [ ]:
# Create a function to deploy a model to a SageMaker endpoint
def deploy_model_to_endpoint(model_path, model_name, instance_type="ml.t3.medium"):
    """Deploy a model to a SageMaker endpoint.
    
    Args:
        model_path (str): S3 path to the model artifacts
        model_name (str): Name for the model and endpoint
        instance_type (str): Instance type for the endpoint
        
    Returns:
        predictor: SageMaker predictor for the endpoint
    """
    from sagemaker.huggingface import HuggingFaceModel
    
    # Create a unique endpoint name
    endpoint_name = f"{model_name.replace('/', '-')}-{int(time.time())}"[-63:]
    
    # Create the model
    huggingface_model = HuggingFaceModel(
        model_data=model_path,
        role=SAGEMAKER_ROLE_ARN,
        transformers_version="4.26",
        pytorch_version="1.13",
        py_version="py39"
    )
    
    # Deploy the model
    predictor = huggingface_model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )
    
    print(f"Deployed model to endpoint: {endpoint_name}")
    return predictor

## 11. Next Steps

Now that we've applied knowledge distillation to our models and tested their inference performance, we'll explore how to deploy these optimized models in production environments in the next notebook.

### What We've Learned:
- How to apply knowledge distillation to transformer models
- How to use SageMaker Processing for distributed optimization tasks
- How knowledge distillation affects model size and inference speed
- The real-world performance benefits of distilled models

### What's Next - Model Hosting:
In the next notebook, we'll explore different options for hosting our optimized models, including SageMaker endpoints, SageMaker multi-model endpoints, and serverless inference. We'll also compare the cost implications of different hosting options and discuss best practices for deploying optimized models in production.